<a href="https://colab.research.google.com/github/samhoon000/Real-World-Drug-Surveillance/blob/main/Real_World_Drug_Surveillance.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import requests
all_data=[]
for i in range(0,5000,100):
  url ="https://api.fda.gov/drug/event.json?limit=100"
  response=requests.get(url)
  if response.status_code !=200:
    print("Stopped at: ",i)
    break
  data=response.json()
  all_data.extend(data['results'])

In [5]:
import pandas as pd
df_uncleaned=pd.json_normalize(all_data)

In [6]:
df_uncleaned.head()

,safetyreportid,transmissiondateformat,transmissiondate,serious,seriousnessdeath,receivedateformat,receivedate,receiptdateformat,receiptdate,fulfillexpeditecriteria,...,sender.sendertype,receiver.receivertype,receiver.receiverorganization,seriousnessother,occurcountry,patient.patientagegroup,seriousnesshospitalization,patient.summary.narrativeincludeclinical,seriousnesslifethreatening,patient.patientweight
0,5801206-7,102,20090109,1,1,102,20080707,102,20080625,1,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,10003300,102,20141002,1,NaN,102,20140306,102,20140306,2,...,2,6,FDA,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,10003301,102,20141002,1,NaN,102,20140228,102,20140228,2,...,2,6,FDA,1,NaN,NaN,NaN,NaN,NaN,NaN
3,10003302,102,20141002,2,NaN,102,20140312,102,20140312,2,...,2,6,FDA,NaN,US,NaN,NaN,NaN,NaN,NaN
4,10003304,102,20141212,2,NaN,102,20140312,102,20140424,2,...,2,6,FDA,NaN,US,NaN,NaN,NaN,NaN,NaN


In [13]:
def clean_data(df_uncleaned):
  #Keep the necessary columns
  df=df_uncleaned[[
      'serious',
      'patient.patientonsetage',
      'patient.reaction',
      'patient.drug',
      'occurcountry']].copy()

  # Extract reactions of a patient
  df['reaction_clean']=df['patient.reaction'].apply( lambda lst: [x.get('reactionmeddrapt') for x in lst if 'reactionmeddrapt' in x] if isinstance(lst,list) else [])

  #Extract name of the disease
  df['drug_name']=df['patient.drug'].apply(lambda lst:[x.get('medicinalproduct') for x in lst if 'medicinalproduct' in x ] if isinstance(lst,list) else [])

  #Drop original columns
  df=df.drop(columns=['patient.reaction','patient.drug'])

  #Explode lists → one row per reaction and drug
  df=df.explode('reaction_clean')
  df=df.explode('drug_name')

  #Convert datatype
  df['serious']=pd.to_numeric(df['serious'],errors='coerce')
  df['patient.patientonsetage']=pd.to_numeric(df['patient.patientonsetage'],errors='coerce')

  #Rename columns
  df.rename(columns={'reaction_clean':'reactions','patient.patientonsetage':'patient_age'},inplace=True)

  #Drop Missing values
  df=df.dropna()

  #Reset index values
  df = df.reset_index(drop=True)

  return df

In [14]:
clean_data(df_uncleaned)

,serious,patient_age,occurcountry,reactions,drug_name
0,2,48.0,US,Drug hypersensitivity,LIPITOR
1,2,48.0,US,Drug hypersensitivity,LISINOPRIL
2,2,48.0,US,Drug hypersensitivity,LOSARTAN POTASSIUM
3,2,48.0,US,Drug hypersensitivity,METOPROLOL TARTRATE
4,2,48.0,US,Drug hypersensitivity,MINOCYCLINE
...,...,...,...,...,...
39645,2,56.0,US,Dizziness exertional,LETAIRIS
39646,2,56.0,US,Haemoglobin decreased,LETAIRIS
39647,2,56.0,US,Dyspnoea,LETAIRIS
39648,2,65.0,US,Hot flush,LETAIRIS
